# Notebook 10 — Sensitivity Analysis and Bifurcation

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 10 of 12  
**Time estimate:** 75 minutes

> A model has many parameters. Which ones actually matter? Sensitivity analysis
> tells you which parameters most influence the output. Bifurcation diagrams show
> how the system's long-run behaviour changes qualitatively as a single parameter
> is varied — revealing thresholds, switches, and bistability.

---
## Step 1 — Motivation

A genetic regulatory network has 20 kinetic parameters, all with uncertainty.
Which ones must be measured accurately? Sensitivity analysis reveals the answer:
some parameters contribute little to output variation (can be estimated roughly)
while others are critical (must be measured precisely). Separately, bifurcation
analysis reveals that the same network can have two stable steady states — a
switch. A cell uses this to make a commitment: differentiate or stay progenitor.
Once committed, the switch is irreversible even if the inducing signal is removed.

---
## Step 2 — Intuition

**Sensitivity analysis:**
- *One-at-a-time (OAT):* vary each parameter ±10%, measure output change.
  Fast but misses interactions between parameters.
- *Morris elementary effects:* vary parameters one at a time, but starting from
  many different points in parameter space. More robust than OAT.
- *Sobol indices (variance-based):* decompose total output variance into
  contributions from each parameter and their interactions.
  First-order Sobol index S_i = fraction of variance explained by parameter i alone.
  Total-order S_Ti = fraction explained by i plus all interactions involving i.

**Bifurcation diagram:** plot the long-run steady state (or equilibrium) of
the system as a function of a control parameter. Three key features:
- **Saddle-node bifurcation:** two stable steady states collide and annihilate.
  At the bifurcation point, one stable branch and one unstable branch meet.
- **Bistability:** a range of the control parameter where two stable states coexist
  (separated by an unstable state). The system exhibits hysteresis.
- **Hopf bifurcation:** a stable fixed point loses stability and becomes a limit
  cycle (oscillations emerge) — relevant for circadian rhythms, NF-κB oscillations.

---
## Step 3 — Biological Background

**Bistability in biology:**
- *Cell fate decision:* the GATA1/PU.1 toggle switch in hematopoiesis produces
  either erythrocytes (GATA1 dominant) or myeloid cells (PU.1 dominant).
  The bistable switch is implemented by mutual repression.
- *Cell cycle commitment:* the G1/S transition is a saddle-node switch — once
  CDK2 activity crosses a threshold, Rb is phosphorylated and the cell commits
  to dividing irreversibly.
- *Positive feedback + bistability = memory:* lactose operon in E. coli uses
  bistability: either all-on (lactose present) or all-off. Intermediate states
  are unstable. This implements epigenetic memory across cell divisions.

**Logistic map and chaos:** $x_{n+1} = rx_n(1-x_n)$. Period doubling as r
increases → chaos at r ≈ 3.57. A minimal example of how a simple deterministic
rule can produce unpredictable dynamics — relevant for population ecology (boom-bust
cycles) and turbulence.

---
## Step 4 — Mathematical Explanation

**Bistable ODE (mutual repression toggle switch):**
$$\dot{u} = \frac{\alpha}{1 + v^n} - u, \quad \dot{v} = \frac{\alpha}{1 + u^n} - v$$
Fixed points satisfy $u^* = \alpha/(1 + v^{*n})$. For $n > 1$ and α large enough,
three fixed points exist: two stable (high-u/low-v and low-u/high-v) and one
unstable (symmetric). The separatrix between the two stable basins passes through
the unstable fixed point.

**Sobol indices (variance decomposition):**
$$\text{Var}(Y) = \sum_i V_i + \sum_{i<j} V_{ij} + \ldots$$
$$S_i = V_i / \text{Var}(Y), \quad S_{Ti} = (V_i + \sum_{j\neq i} V_{ij} + \ldots) / \text{Var}(Y)$$
Estimated by Monte Carlo: requires 2*(n_params+1)*N model evaluations.

**Saddle-node bifurcation:** at the bifurcation point, the Jacobian has an
eigenvalue of 0 — the system is at the border of stability. Normal form:
$\dot{x} = r + x^2$ — for r < 0, two fixed points; at r=0, they collide;
for r > 0, none.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

rng = np.random.default_rng(42)

# ---- Mutual repression toggle switch ----
def toggle_rhs(t, y, alpha, n):
    u, v = y
    du = alpha / (1 + v**n) - u
    dv = alpha / (1 + u**n) - v
    return [du, dv]

def find_steady_states(alpha, n=2, n_ics=50):
    """Find all stable steady states by integrating from many initial conditions."""
    ss_list = []
    for u0 in np.linspace(0, alpha, n_ics):
        v0 = alpha - u0  # roughly sum = alpha
        sol = solve_ivp(toggle_rhs, (0, 200), [u0, v0],
                         args=(alpha, n), method='RK45', max_step=0.5)
        ss = sol.y[:, -1]
        # Round to avoid duplicates
        ss_round = tuple(np.round(ss, 2))
        if ss_round not in [tuple(np.round(s, 2)) for s in ss_list]:
            ss_list.append(ss)
    return ss_list

# Bifurcation diagram: vary alpha, find steady states
alpha_vals = np.linspace(0.5, 8.0, 60)
bif_u_high, bif_u_low, bif_u_mid = [], [], []

for alpha in alpha_vals:
    ss = find_steady_states(alpha)
    u_vals_ss = sorted([s[0] for s in ss])
    if len(u_vals_ss) >= 2:
        bif_u_high.append((alpha, max(u_vals_ss)))
        bif_u_low.append((alpha, min(u_vals_ss)))
    elif len(u_vals_ss) == 1:
        bif_u_high.append((alpha, u_vals_ss[0]))
        bif_u_low.append((alpha, u_vals_ss[0]))

print(f'Toggle switch bifurcation: computed {len(alpha_vals)} alpha values')

# ---- Logistic map: period-doubling route to chaos ----
def logistic_attractor(r, n_transient=500, n_collect=100):
    x = 0.5
    for _ in range(n_transient):
        x = r * x * (1 - x)
    attractor = []
    for _ in range(n_collect):
        x = r * x * (1 - x)
        attractor.append(x)
    return attractor

r_vals_logistic = np.linspace(2.5, 4.0, 1000)
logistic_bif = []
for r in r_vals_logistic:
    attrs = logistic_attractor(r)
    for x in attrs:
        logistic_bif.append((r, x))
print(f'Logistic map: {len(logistic_bif)} points computed')

# ---- SIR sensitivity analysis (Morris-like OAT) ----
def sir_peak_infected(beta, gamma, N=10000, I0=5):
    sol = solve_ivp(
        lambda t, y: [-beta*y[0]*y[1]/N, beta*y[0]*y[1]/N - gamma*y[1], gamma*y[1]],
        (0, 300), [N-I0, I0, 0], max_step=0.5
    )
    return sol.y[1].max() / N

# Parameter ranges: beta in [0.1, 0.5], gamma in [0.05, 0.25]
params_nom = {'beta': 0.3, 'gamma': 0.1}
params_range = {'beta': (0.1, 0.5), 'gamma': (0.05, 0.25)}

# OAT sensitivity: vary each ±50% from nominal
oat_results = {}
for param, (lo, hi) in params_range.items():
    vals = np.linspace(lo, hi, 20)
    outputs = []
    for v in vals:
        p = params_nom.copy(); p[param] = v
        outputs.append(sir_peak_infected(p['beta'], p['gamma']))
    oat_results[param] = (vals, np.array(outputs))

print('OAT sensitivity analysis done.')

# Variance-based sensitivity (Monte Carlo with N_SAMPLES)
N_SAMPLES = 200
beta_mc  = rng.uniform(0.1, 0.5, N_SAMPLES)
gamma_mc = rng.uniform(0.05, 0.25, N_SAMPLES)
Y_all   = np.array([sir_peak_infected(b, g) for b, g in zip(beta_mc, gamma_mc)])
V_total = Y_all.var()

# Conditional variance: fix beta (first-order Sobol S_beta)
beta_bins = np.percentile(beta_mc, np.linspace(0, 100, 11))
V_beta_cond = []
for i in range(len(beta_bins)-1):
    mask = (beta_mc >= beta_bins[i]) & (beta_mc < beta_bins[i+1])
    if mask.sum() > 2:
        V_beta_cond.append(np.mean(Y_all[mask]))
V_beta_cond = np.array(V_beta_cond)
S_beta_approx = V_beta_cond.var() / V_total if V_total > 0 else 0
print(f'Approximate first-order Sobol S_beta={S_beta_approx:.3f}')
print(f'(beta explains ~{S_beta_approx*100:.0f}% of output variance)')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel A: Toggle switch trajectories
ax = axes[0, 0]
for u0, col in [(0.1, 'steelblue'), (3.9, 'tomato'), (2.0, 'green')]:
    sol = solve_ivp(toggle_rhs, (0, 20), [u0, 4-u0], args=(4.0, 2), max_step=0.05)
    ax.plot(sol.t, sol.y[0], color=col, lw=2, label=f'u₀={u0:.1f}')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Time'); ax.set_ylabel('u(t) (gene A expression)')
ax.set_title('A. Toggle switch: bistability\n(two stable states depending on IC)')
ax.legend(fontsize=8)

# Panel B: Toggle bifurcation diagram
ax = axes[0, 1]
if bif_u_high:
    ah, uh = zip(*bif_u_high)
    al, ul = zip(*bif_u_low)
    ax.plot(ah, uh, 'steelblue', lw=2, label='Stable (high-u)')
    ax.plot(al, ul, 'tomato', lw=2, label='Stable (low-u)')
    # Mark bistable region
    ah_arr, uh_arr = np.array(ah), np.array(uh)
    al_arr, ul_arr = np.array(al), np.array(ul)
    mask = np.abs(uh_arr - ul_arr) > 0.5
    if mask.any():
        ax.fill_between(ah_arr[mask], ul_arr[mask], uh_arr[mask],
                          alpha=0.15, color='grey', label='Bistable region')
ax.set_xlabel('Production rate α'); ax.set_ylabel('Steady-state u*')
ax.set_title('B. Bifurcation diagram: toggle switch\n(vs. production rate α)')
ax.legend(fontsize=8)

# Panel C: Logistic map bifurcation
ax = axes[0, 2]
if logistic_bif:
    r_bif, x_bif = zip(*logistic_bif)
    ax.scatter(r_bif, x_bif, s=0.1, color='steelblue', alpha=0.6)
ax.set_xlabel('Growth rate r'); ax.set_ylabel('Attractor x')
ax.set_title('C. Logistic map bifurcation diagram\n(period-doubling → chaos at r≈3.57)')
for r_mark, label in [(3.0, 'Period 2'), (3.449, 'Period 4'), (3.544, 'Period 8'), (3.57, 'Chaos')]:
    ax.axvline(r_mark, color='tomato', ls=':', lw=0.8, alpha=0.7)
    ax.text(r_mark, 0.95, label, rotation=90, fontsize=6, va='top', ha='right')

# Panel D: OAT sensitivity (beta)
ax = axes[1, 0]
for param, col, ls in [('beta', 'steelblue', '-'), ('gamma', 'tomato', '--')]:
    vals, outs = oat_results[param]
    # Normalize to [0, 1] for comparison
    ax.plot(vals / vals.max(), outs, color=col, lw=2, ls=ls, label=param)
ax.set_xlabel('Normalized parameter value')
ax.set_ylabel('Peak infected fraction')
ax.set_title('D. OAT sensitivity: SIR peak infection\n(β has larger effect than γ)')
ax.legend(fontsize=9)

# Panel E: 2D sensitivity heatmap
ax = axes[1, 1]
beta_grid  = np.linspace(0.1, 0.5, 15)
gamma_grid = np.linspace(0.05, 0.25, 15)
Z = np.zeros((15, 15))
for i, b in enumerate(beta_grid):
    for j, g in enumerate(gamma_grid):
        Z[j, i] = sir_peak_infected(b, g)
im = ax.contourf(beta_grid, gamma_grid, Z, levels=15, cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Peak infected fraction')
ax.set_xlabel('β (transmission rate)')
ax.set_ylabel('γ (recovery rate)')
ax.set_title('E. 2D sensitivity: SIR peak infection\n(joint β × γ effect)')
ax.plot(BETA_TRUE := 0.3, GAMMA_TRUE := 0.1, 'w*', ms=12, label='Nominal')
ax.legend(fontsize=8)

# Panel F: Phase portrait for toggle switch
ax = axes[1, 2]
u_range = np.linspace(0.01, 5, 30)
v_range = np.linspace(0.01, 5, 30)
U, V = np.meshgrid(u_range, v_range)
alpha_phase = 4.0; n_phase = 2
dU = alpha_phase / (1 + V**n_phase) - U
dV = alpha_phase / (1 + U**n_phase) - V
ax.streamplot(u_range, v_range, dU, dV, color='grey', density=1.2, linewidth=0.8)
# Nullclines
u_null = alpha_phase / (1 + v_range**n_phase)
v_null = alpha_phase / (1 + u_range**n_phase)
ax.plot(u_null, v_range, 'steelblue', lw=2, label='u-nullcline')
ax.plot(u_range, v_null, 'tomato', lw=2, label='v-nullcline')
ax.set_xlabel('u'); ax.set_ylabel('v')
ax.set_title('F. Toggle switch phase portrait\n(α=4, n=2 — three fixed points)')
ax.set_xlim(0, 5); ax.set_ylim(0, 5)
ax.legend(fontsize=8)

plt.suptitle('Module 15 NB10: Sensitivity Analysis and Bifurcation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('sensitivity_bifurcation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Draw the bifurcation diagram for the logistic population model
   dN/dt = rN(1-N/K) as a function of K (carrying capacity). What type of
   bifurcation occurs at K=0? Show the stable branch and the unstable branch.
2. Compute the Sobol first-order and total-order indices for the SIR model
   using the Saltelli estimator (scipy or from scratch). Compare to the
   OAT sensitivity — do they agree on which parameter is more important?
3. Add a third gene (Z) to the toggle switch that activates U but represses V.
   Show the bifurcation diagram — does the tristability emerge?

---
## Step 10 — Quiz

1. What is a saddle-node bifurcation? Describe what happens to the fixed points
   of the system as the control parameter crosses the bifurcation point.
2. What is hysteresis in a bistable system? Describe a biological example where
   hysteresis is functionally important.
3. What is the difference between a first-order Sobol index and a total-order
   Sobol index? When do they differ significantly?
4. In the logistic map, what is the Feigenbaum constant δ ≈ 4.67 and what does
   it tell you about the rate of period-doubling bifurcations?

---
## Step 12 — Reflection

> *[In one paragraph: explain why bistability is biologically useful for cell fate
> decisions. Start from: "imagine a stem cell deciding between two fates..." Include
> the role of hysteresis in making the decision irreversible. Why would a cell that
> commits reversibly (no bistability) be problematic for an organism?]*

---
*Next: `11_multiscale_modeling.ipynb`*